# Partycjonowanie i skalowanie danych — benchmark A/B/C

A: jeden plik Parquet. B: zrównoważone grupy stacji. C: partycje według stacji.
Każdy scenariusz: A, B, B bez filtra, C, C bez filtra. Przyspieszenia tylko dla zgodnych zakresów.

Ustaw `SCALE_FACTORS = [1]` dla oryginału, `[10]` dla dodatkowego testu lub `[1, 10]` dla obu przebiegów i porównania. Każdy przebieg ma osobny katalog. SILVER pozostaje bez zmian.

Skala dotyczy liczby rekordów, nie liczby stacji ani rozmiaru pliku. Kopie mają rozłączne bloki czasu UTC; inne cechy nie są przeliczane. Dane syntetyczne służą wyłącznie testom wydajności, nie treningowi.

## 1. Konfiguracja
Wszystkie ustawienia eksperymentu są w poniższej komórce. Źródło przygotuj wcześniej notebookiem 05. Zapis A/B/C i walidacja wymagają RAM; limit DuckDB nie ogranicza pamięci Polars/PyArrow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import runpy
import json
import polars as pl
from IPython.display import display, Markdown

PROJECT_ROOT = None  # np. r"D:/gios-bigdata"; None = automatyczne wyszukiwanie
SOURCE_PATH = "data/silver/pm25_pogoda_2024.parquet"  # względem projektu lub absolutna
SCALE_FACTORS = [1, 10]  # [1], [10] lub [1, 10]
RUN_LABEL = "partition_test"
GROUP_COUNT = 8
SCENARIO_SIZES = [1, 8, 32]  # dodatkowo zawsze wszystkie stacje
REPEATS = 20
WARMUPS = 3
THREADS = None  # None = maksymalnie 4
ROW_GROUP_SIZE = 122880
WRITE_STATISTICS = False
READER_MODE = "all_files_list"  # lub "glob"
SEED = 42
SMALL_DIFFERENCE_PERCENT = 5.0
MEMORY_LIMIT = None  # np. "4GB"; dotyczy tylko DuckDB
SHOW_PLOTS = True

## 2. Projekt i kontrola konfiguracji

In [ ]:
def find_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts/partition_benchmark.py").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono scripts/partition_benchmark.py")

ROOT = find_root(PROJECT_ROOT if PROJECT_ROOT is not None else Path.cwd())
benchmark = runpy.run_path(str(ROOT / "scripts/partition_benchmark.py"))
if not SCALE_FACTORS or any(type(n) is not int or n < 1 for n in SCALE_FACTORS):
    raise ValueError("SCALE_FACTORS musi zawierać dodatnie liczby całkowite")
if len(set(SCALE_FACTORS)) != len(SCALE_FACTORS):
    raise ValueError("Powtórzone skale")
source = Path(SOURCE_PATH)
source = source if source.is_absolute() else ROOT / source
if not source.is_file():
    raise FileNotFoundError(f"Najpierw wykonaj notebook 05: {source}")

SETTINGS = dict(root=ROOT, source_path=source, repeats=REPEATS, warmups=WARMUPS,
    threads=THREADS, group_count=GROUP_COUNT, row_group_size=ROW_GROUP_SIZE,
    write_statistics=WRITE_STATISTICS, reader_mode=READER_MODE, seed=SEED,
    scenario_sizes=SCENARIO_SIZES, small_difference_percent=SMALL_DIFFERENCE_PERCENT,
    memory_limit=MEMORY_LIMIT, run_label=RUN_LABEL)
print("Projekt:", ROOT)
print("Źródło:", source)
print("Skale:", SCALE_FACTORS)

## 3. Uruchomienie
Generowanie syntetycznego źródła, zapis A/B/C, pełna kontrola danych oraz plany zapytań są poza timerem. Pomiar obejmuje execute + fetchall. Cache systemowy nie jest czyszczony, a walidacja go rozgrzewa. Kolejność scenariuszy i wariantów jest losowana. Przebiegi skal wykonywane są kolejno, więc porównanie nie jest testem zimnego cache.

Raporty zawierają konfigurację środowiska, parametry skalowania, walidację, plany, SQL, surowe pomiary, podsumowanie, porównania, strukturę plików i wykresy. Nie przerywaj komórki po ukończeniu pierwszej skali, jeżeli chcesz otrzymać oba wyniki.

In [ ]:
results = []
for scale in SCALE_FACTORS:
    print(f"Start: skala {scale}×")
    result = benchmark["run_experiment"](scale_factor=scale, **SETTINGS)
    results.append(result)
    print("Wiersze:", result["scaling"]["actual_rows"])
    print("Raport:", result["report"])
    display(result["summary"])
    display(result["storage"])

## 4. Łączne wyniki i porównanie ze skalą 1×

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid4().hex[:8]
suite_report = ROOT / "reports/partition_benchmark" / f"suite_{stamp}"
suite_report.mkdir(parents=True, exist_ok=False)
combined = pl.concat([
    r["summary"].with_columns(pl.lit(r["scale_factor"]).alias("scale_factor"))
    for r in results
])
combined.write_csv(suite_report / "all_scales_summary.csv")
all_storage = pl.concat([
    r["storage"].with_columns(pl.lit(r["scale_factor"]).alias("scale_factor"))
    for r in results
])
all_storage.write_csv(suite_report / "all_scales_storage.csv")
display(combined)

if 1 in SCALE_FACTORS and len(SCALE_FACTORS) > 1:
    keys = ["id", "scenario", "variant", "scope"]
    reference = combined.filter(pl.col("scale_factor") == 1).select(
        *keys, pl.col("median_ms").alias("median_ms_1x"),
        pl.col("mean_ms").alias("mean_ms_1x")
    )
    scaling_comparison = (
        combined.filter(pl.col("scale_factor") != 1)
        .join(reference, on=keys, how="left", validate="m:1")
        .with_columns(
            (pl.col("median_ms") / pl.col("median_ms_1x")).alias("median_time_ratio"),
            (pl.col("mean_ms") / pl.col("mean_ms_1x")).alias("mean_time_ratio")
        )
    )
    scaling_comparison.write_csv(suite_report / "scaling_comparison.csv")
    display(scaling_comparison)
else:
    print("Porównanie między skalami wymaga 1 i co najmniej jednej większej skali w SCALE_FACTORS.")

manifest = [{"scale_factor": r["scale_factor"], "report": str(r["report"]),
             "data": str(r["data"]), "scaling": r["scaling"]} for r in results]
(suite_report / "runs.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Raport zbiorczy:", suite_report)

## 5. Interpretacja i wykresy

In [ ]:
for result in results:
    display(Markdown(f"## Skala {result['scale_factor']}×"))
    display(Markdown((result["report"] / "interpretation.md").read_text(encoding="utf-8")))
    if SHOW_PLOTS:
        from IPython.display import Image
        for path in sorted(result["report"].glob("*.png")):
            display(Image(filename=str(path)))

## Ograniczenia interpretacji
- 10× więcej wierszy nie musi oznaczać 10× większych plików ani 10× dłuższego czasu.
- Liczba stacji i partycji C pozostaje taka sama; badamy wzrost liczby rekordów na stację.
- Warianty bez filtra porównujemy z wariantami wykonującymi pełną agregację, nigdy z pracą ograniczoną do podzbioru stacji.
- Raport scaling_comparison.csv porównuje identyczny scenariusz i wariant między skalami; median_time_ratio oznacza czas danej skali podzielony przez czas 1×.
- Kopiowane cechy kalendarzowe i historyczne nie opisują nowych dat: to syntetyczny test objętości, nie nowe obserwacje GIOŚ.
- Rozmiar zapisanych plików nie jest miarą bajtów fizycznie odczytanych podczas zapytania.
